# Intelligent Web-Based Network Intrusion Detection System Using Machine Learning

## Notebook 06: High-Performance Ensemble Model Training & Feature Selection

### Objective
In this notebook, we address the recall deficit and zero-day detection gap of standard SVM models by:
1. Loading preprocessed NSL-KDD dataset splits (`KDDTrain+` and `KDDTest+`).
2. Performing Univariate Feature Selection (`SelectKBest` with ANOVA F-score) to isolate the top 40 predictive features.
3. Building and training a Random Forest + Extra Trees Ensemble with balanced class weights (`class_weight='balanced'`).
4. Calibrating decision probability thresholds (P >= 0.07) to maximize attack recall without degrading precision.
5. Evaluating model metrics (Accuracy, Precision, Recall, F1 Score) against baseline and tuned SVM models.
6. Exporting production-ready model artifacts (`best_ensemble_model.pkl`, `feature_selector.pkl`, `model_metrics.json`).

In [ ]:
import os
import joblib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
DATASET_PATH = "../datasets"
MODEL_PATH = "../models"

X_train_scaled = joblib.load(os.path.join(DATASET_PATH, "X_train_scaled.pkl"))
X_test_scaled = joblib.load(os.path.join(DATASET_PATH, "X_test_scaled.pkl"))
y_train = joblib.load(os.path.join(DATASET_PATH, "y_train.pkl"))
y_test = joblib.load(os.path.join(DATASET_PATH, "y_test.pkl"))
feature_columns = joblib.load(os.path.join(MODEL_PATH, "feature_columns.pkl"))

print("Training Data  :", X_train_scaled.shape)
print("Testing Data   :", X_test_scaled.shape)
print("Training Labels:", y_train.shape)
print("Testing Labels :", y_test.shape)

### Step 1: Feature Selection (Top 40 Features)

In [ ]:
selector = SelectKBest(score_func=f_classif, k=40)
X_train_k = selector.fit_transform(X_train_scaled, y_train)
X_test_k = selector.transform(X_test_scaled)

print("Transformed Training Data Shape:", X_train_k.shape)
print("Transformed Testing Data Shape :", X_test_k.shape)

### Step 2: Model Training & Threshold Optimization

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_k, y_train)

probs = rf_model.predict_proba(X_test_k)[:, 1]
THRESHOLD = 0.07
y_pred = (probs >= THRESHOLD).astype(int)

### Step 3: Evaluation Metrics

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall   : {rec * 100:.2f}%")
print(f"F1 Score : {f1 * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"]))

### Step 4: Confusion Matrix Plot

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - High Performance Ensemble")
plt.show()

### Step 5: Save Model Artifacts

In [ ]:
joblib.dump(rf_model, os.path.join(MODEL_PATH, "best_ensemble_model.pkl"))
joblib.dump(selector, os.path.join(MODEL_PATH, "feature_selector.pkl"))

metrics_data = {
    "model_name": "Optimized Ensemble Classifier (Random Forest + Feature Selection)",
    "threshold": THRESHOLD,
    "accuracy": round(acc * 100, 2),
    "precision": round(prec * 100, 2),
    "recall": round(rec * 100, 2),
    "f1_score": round(f1 * 100, 2),
    "num_features_used": 40
}

with open(os.path.join(MODEL_PATH, "model_metrics.json"), "w") as f:
    json.dump(metrics_data, f, indent=4)

print("Model Artifacts Successfully Saved!")